# Generation of a simple DFN with PorePy and Transfer to OGS

This is work in progress

In [1]:
import numpy as np
import porepy as pp
import ogstools as ogs
import os as os

/home/mok/build/release2/.venv/lib/python3.12/site-packages/porepy/numerics/nonlinear/nonlinear_solvers.py:14: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import trange  # type: ignore


# Setting up the domain and generating a random set of circular fractures

In [2]:
mins = np.array([0.,0.,0.])
maxs = np.array([10.,10.,10.])

In [3]:
bounding_box = {'xmin': mins[0], 'xmax': maxs[0], 'ymin': mins[1], 'ymax': maxs[1], 'zmin': mins[2], 'zmax': maxs[2]}
domain = pp.Domain(bounding_box=bounding_box)
domain

pp.Domain(bounding_box={'xmin': 0.0, 'xmax': 10.0, 'ymin': 0.0, 'ymax': 10.0, 'zmin': 0.0, 'zmax': 10.0})

In [4]:
nfracs = 8
r_range = np.array([3,9])
f_i = np.array([])
for i in range(nfracs):
    center = np.random.rand(3) * (maxs - mins) + mins
    major_axis = np.random.rand() * (r_range[1] - r_range[0]) + r_range[0]
    minor_axis = major_axis.copy() #circular
    major_axis_angle = 0. #for circular
    strike_angle = np.random.rand() * np.pi - np.pi/2
    dip_angle = np.random.rand() * np.pi - np.pi/2
    f_i = np.append(f_i,pp.create_elliptic_fracture(center, major_axis, minor_axis, major_axis_angle, strike_angle, dip_angle))

In [5]:
network = pp.create_fracture_network(fractures=f_i,domain=domain)
network

Three-dimensional fracture network with 8 plane fractures.
The domain is a cuboid with bounding box: {'xmin': 0.0, 'xmax': 10.0, 'ymin': 0.0, 'ymax': 10.0, 'zmin': 0.0, 'zmax': 10.0}.

## Meshing ... 

In [6]:
mesh_args = {'cell_size_boundary': 1.0, 'cell_size_fracture': 0.5, 'cell_size_min': 0.2}
mdg = pp.create_mdg("simplex", mesh_args, network)

In [7]:
#Removal of 3D not needed really
mdg2d = mdg.copy()
for sd in mdg2d.subdomains():
    if sd.dim == 3:
        mdg2d.remove_subdomain(sd)
mdg2d

Mixed-dimensional grid containing 14 grids and 12 interfaces.
Maximum dimension present: 2 
Minimum dimension present: 1 
8 grids of dimension 2 with in total 5805 cells
6 grids of dimension 1 with in total 46 cells
12 interfaces between grids of dimension 2 and 1 with in total 184 mortar cells.

In [8]:
#pp.plot_grid(mdg2d, figsize=(12,12), plot_2d=False)

## Export to VTU and import in OGS. Setting up Material IDs

In [9]:
pp.Exporter(mdg2d, 'mixed_dimensional_grid').write_vtu()

In [10]:
DFN_2D = ogs.Mesh('mixed_dimensional_grid_constant_2.vtu')
DFN_2D

Mesh (0x74c177c7a020)
  N Cells:    5805
  N Points:   3266
  X Bounds:   0.000e+00, 1.000e+01
  Y Bounds:   0.000e+00, 1.000e+01
  Z Bounds:   0.000e+00, 1.000e+01
  N Arrays:   6

In [11]:
DFN_2D['MaterialIDs'] = DFN_2D['subdomain_id'] - DFN_2D['subdomain_id'].min()

In [12]:
fig = DFN_2D.plot('MaterialIDs',show_edges=True)

Widget(value='<iframe src="http://localhost:45737/index.html?ui=P_0x74c177c751f0_0&reconnect=auto" class="pyvi…

## Generating boundaries for OGS

In [13]:
cmd = '~/build/release2/bin/ExtractBoundary -i mixed_dimensional_grid_constant_2.vtu -o boundaries.vtu'
os.system(cmd)

[2025-03-29 14:44:05.805] [ogs] [info] Mesh read: 3266 nodes, 5805 elements.
[2025-03-29 14:44:05.806] [ogs] [info] 6 property vectors copied, 0 vectors skipped.
[2025-03-29 14:44:05.806] [ogs] [info] Created surface mesh: 703 nodes, 703 elements.


0

In [14]:
tol = 1e-3
cmd = '~/build/release2/bin/removeMeshElements -i boundaries.vtu -o xmax.vtu --x-max %.3f' %(maxs[0]-tol)
os.system(cmd)

cmd = '~/build/release2/bin/removeMeshElements -i boundaries.vtu -o xmin.vtu --x-min %.3f' %(mins[0]+tol)
os.system(cmd)

cmd = '~/build/release2/bin/removeMeshElements -i boundaries.vtu -o ymax.vtu --y-max %.3f' %(maxs[1]-tol)
os.system(cmd)

cmd = '~/build/release2/bin/removeMeshElements -i boundaries.vtu -o ymin.vtu --y-min %.3f' %(mins[1]+tol)
os.system(cmd)

cmd = '~/build/release2/bin/removeMeshElements -i boundaries.vtu -o zmax.vtu --z-max %.3f' %(maxs[2]-tol)
os.system(cmd)

cmd = '~/build/release2/bin/removeMeshElements -i boundaries.vtu -o zmin.vtu --z-min %.3f' %(mins[2]+tol)
os.system(cmd)

[2025-03-29 14:44:06.245] [ogs] [info] Mesh read: 703 nodes, 703 elements.
[2025-03-29 14:44:06.245] [ogs] [info] Bounding box of "boundaries" is
x = [0.000000,10.000000]
y = [0.000000,10.000000]
z = [0.000000,10.000000]
[2025-03-29 14:44:06.245] [ogs] [info] 677 elements found.
[2025-03-29 14:44:06.245] [ogs] [info] Removing total 677 elements...
[2025-03-29 14:44:06.245] [ogs] [info] 26 elements remain in mesh.
[2025-03-29 14:44:06.245] [ogs] [info] Removing total 675 nodes...
[2025-03-29 14:44:06.272] [ogs] [info] Mesh read: 703 nodes, 703 elements.
[2025-03-29 14:44:06.272] [ogs] [info] Bounding box of "boundaries" is
x = [0.000000,10.000000]
y = [0.000000,10.000000]
z = [0.000000,10.000000]
[2025-03-29 14:44:06.272] [ogs] [info] 651 elements found.
[2025-03-29 14:44:06.272] [ogs] [info] Removing total 651 elements...
[2025-03-29 14:44:06.272] [ogs] [info] 52 elements remain in mesh.
[2025-03-29 14:44:06.272] [ogs] [info] Removing total 646 nodes...
[2025-03-29 14:44:06.305] [ogs] 

0